In [87]:
## Subset data from Cao et al. to only haematopoietic cells
suppressPackageStartupMessages({
  library(dplyr)
  library(data.table)
  library(R.utils)
  library(Matrix)
})
main = '/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/schendure/'

clusters_keep = c('Primitive erythroid lineage',
                  'Definitive erythroid lineage', 
                  'Megakaryocytes', 
                  'Endothelial cells')
counts = readRDS(paste0(main, "gene_count_cleaned.RDS"))
meta = fread(paste0(main, 'cell_annotate.csv')) %>% 
    .[,c('sample', 'development_stage', 'Size_Factor','detected_doublet', 'Main_cell_type')] %>%
    .[Main_cell_type %in% clusters_keep & 
      detected_doublet == FALSE &
      sample %in% colnames(counts),]

# 230k cells are way too much, so randomly subset it!
keep = lapply(unique(meta$development_stage), function(x){
    if(sum(meta$development_stage == x) < 5000) {
    return(which(meta$development_stage == x))
  } else {
    hits = which(meta$development_stage == x)
    return(sample(hits, 5000))
  }
})
keep = do.call(c, keep)

meta = meta[keep,]

nrow(meta)

counts = counts[, meta$sample]


[1] 25000

In [108]:
summary(colnames(counts) == meta$sample) 

   Mode    TRUE 
logical   25000 

In [97]:
# Select genes
genes = fread(paste0(main, 'GSE119945_gene_annotate.csv')) %>%
    .[gene_type=='protein_coding',]

In [95]:
counts = counts[genes$gene_id,]
rownames(counts) = genes$gene_short_name

In [100]:
counts[1:10, 1:10]

   [[ suppressing 10 column names ‘sci3-me-409.GTATCGCATCCGCTCCGGC’, ‘sci3-me-459.GTCGGAGTTTAGACTTCTT’, ‘sci3-me-707.TTCCATCTTTCCGTTCCTTA’ ... ]]



10 x 10 sparse Matrix of class "dgCMatrix"
                           
Xkr4    . . . . . . . . . .
Rp1     . . . . . . 1 . . .
Sox17   . . . . . . . . . .
Mrpl15  . . . . . . . . . .
Lypla1  . . . . . . . . . .
Gm37988 . . . . 1 . . . . .
Tcea1   . . . . . . . . . .
Rgs20   . . . . . . . . . .
Atp6v1h . . . . . . . . . .
Oprk1   . . . . . . . . . .

In [101]:
writeMM(counts, paste0(main, 'raw_counts.mtx'))

NULL

In [106]:
write.csv(rownames(counts), paste0(main, 'genes.csv'), row.names=FALSE)
write.csv(colnames(counts), paste0(main, 'cells.csv'), row.names=FALSE)

# Test

In [26]:
suppressPackageStartupMessages({
    library(SingleCellExperiment)
    library(data.table)
    library(Matrix)
    library(dplyr)
})

# load cao
cao_in = '/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/schendure/'

counts = readMM(paste0(cao_in, 'raw_counts.mtx')) # readMM for .mtx input
genes = read.csv(paste0(cao_in, 'genes.csv'))[[1]]
cells = read.csv(paste0(cao_in, 'cells.csv'))[[1]]

colnames(counts) = cells
rownames(counts) = genes

In [76]:
meta = fread(paste0(cao_in, 'cell_annotate.csv')) %>% 
    .[sample %in% colnames(counts),] %>%
    .[order(match(sample, colnames(counts)))] %>%
    .[,c('cell', 'stage', 'development_stage', 'exp', 'sample'):= list(sample, paste0('E', development_stage), NULL, 'cao', paste0('cao_', sapply(strsplit(sapply(strsplit(sample,"-"), `[`, 3),"[.]"), `[`, 1)))] %>%
    .[,c('cell', 'sample', 'stage', 'Main_cell_type', 'exp')] %>% 
    setnames('Main_cell_type', 'celltype')

In [82]:
stopifnot(colnames(counts) == meta$cell)

In [79]:
head(colnames(counts))

[1] "sci3-me-409.GTATCGCATCCGCTCCGGC"  "sci3-me-459.GTCGGAGTTTAGACTTCTT" 
[3] "sci3-me-707.TTCCATCTTTCCGTTCCTTA" "sci3-me-180.TTAATGAGCTCCGTTCGGAT"
[5] "sci3-me-570.TTCTGGCCTTTGCGAGGCA"  "sci3-me-580.CTAACGACTTATCATGATC"

In [77]:
head(meta)

cell,sample,stage,celltype,exp
<chr>,<chr>,<chr>,<chr>,<chr>
sci3-me-409.GTATCGCATCCGCTCCGGC,cao_409,E11.5,Endothelial cells,cao
sci3-me-459.GTCGGAGTTTAGACTTCTT,cao_459,E11.5,Primitive erythroid lineage,cao
sci3-me-707.TTCCATCTTTCCGTTCCTTA,cao_707,E11.5,Primitive erythroid lineage,cao
sci3-me-180.TTAATGAGCTCCGTTCGGAT,cao_180,E11.5,Definitive erythroid lineage,cao
sci3-me-570.TTCTGGCCTTTGCGAGGCA,cao_570,E11.5,Definitive erythroid lineage,cao
sci3-me-580.CTAACGACTTATCATGATC,cao_580,E11.5,Definitive erythroid lineage,cao


In [78]:
cao_meta = read.table(paste0(cao_in, 'meta.csv'), header = TRUE, sep = ",", stringsAsFactors = FALSE, comment.char = "$")
head(cao_meta)

,sample,development_stage,Size_Factor,detected_doublet,Main_cell_type,cell_id
,<chr>,<dbl>,<dbl>,<lgl>,<chr>,<int>
1,sci3-me-037.ACGCCATATGAGCATATGG,11.5,0.3185023,FALSE,Primitive erythroid lineage,113922
2,sci3-me-290.AAGGCTACTCATCAGAATG,11.5,0.3199768,FALSE,Definitive erythroid lineage,858173
3,sci3-me-442.TTGAGTCCTTATAGACGCA,11.5,1.5276314,FALSE,Endothelial cells,1296568
4,sci3-me-222.AGTCGCATTTTCTTCCGGT,11.5,4.6507234,FALSE,Endothelial cells,669478
5,sci3-me-187.GACTGACGTGAGCATCATT,11.5,3.0125009,FALSE,Definitive erythroid lineage,571245
6,sci3-me-572.TTGCGGTCTTAACTCAATT,11.5,0.8375431,FALSE,Primitive erythroid lineage,1648632


In [57]:
test = meta %>%
    .[,c('cell', 'stage', 'development_stage', 'exp', 'sample'):=
      list(sample, paste0('E', development_stage), NULL, 'cao', paste0('cao_', sapply(strsplit(sapply(strsplit(meta$sample,"-"), `[`, 3),"[.]"), `[`, 1)))] %>%
    .[,c(cell, sample, stage, Main_cell_type, exp)] %>% 
    setnames(c('Main_cell_type'), c('celltype'))
head(test)

ERROR: Error in paste0("E", development_stage): object 'development_stage' not found


In [58]:
head(meta)

sample,Size_Factor,detected_doublet,Main_cell_type,cell,stage,exp
<chr>,<dbl>,<lgl>,<chr>,<chr>,<chr>,<chr>
sci3-me-409.GTATCGCATCCGCTCCGGC,0.6473264,FALSE,Endothelial cells,sci3-me-409.GTATCGCATCCGCTCCGGC,E11.5,cao
sci3-me-459.GTCGGAGTTTAGACTTCTT,1.1825872,FALSE,Primitive erythroid lineage,sci3-me-459.GTCGGAGTTTAGACTTCTT,E11.5,cao
sci3-me-707.TTCCATCTTTCCGTTCCTTA,1.3359402,FALSE,Primitive erythroid lineage,sci3-me-707.TTCCATCTTTCCGTTCCTTA,E11.5,cao
sci3-me-180.TTAATGAGCTCCGTTCGGAT,0.6768174,FALSE,Definitive erythroid lineage,sci3-me-180.TTAATGAGCTCCGTTCGGAT,E11.5,cao
sci3-me-570.TTCTGGCCTTTGCGAGGCA,2.0717395,FALSE,Definitive erythroid lineage,sci3-me-570.TTCTGGCCTTTGCGAGGCA,E11.5,cao
sci3-me-580.CTAACGACTTATCATGATC,1.8343373,FALSE,Definitive erythroid lineage,sci3-me-580.CTAACGACTTATCATGATC,E11.5,cao


In [ ]:
%>% select(cell, sample, stage, Main_cell_type, exp)
colnames(cao_meta) = c('cell', 'sample', 'stage', 'celltype', 'exp')

In [55]:
test = paste0('cao_', sapply(strsplit(sapply(strsplit(meta$sample,"-"), `[`, 3),"[.]"), `[`, 1))

In [56]:
head(test)

[1] "cao_409" "cao_459" "cao_707" "cao_180" "cao_570" "cao_580"

In [ ]:
cao_meta$cell = cao_meta$sample
cao_meta$stage = paste0('E', cao_meta$development_stage)
cao_meta$development_stage = NULL
cao_meta$sample = sapply(strsplit(cao_meta$sample,"-"), `[`, 3)
cao_meta$sample = sapply(strsplit(cao_meta$sample,"[.]"), `[`, 1)
cao_meta$sample = paste0('cao_', cao_meta$sample)
cao_meta$exp = 'cao'

In [44]:
head(meta)

sample,development_stage,Size_Factor,detected_doublet,Main_cell_type
<chr>,<dbl>,<dbl>,<lgl>,<chr>
sci3-me-409.GTATCGCATCCGCTCCGGC,11.5,0.6473264,FALSE,Endothelial cells
sci3-me-459.GTCGGAGTTTAGACTTCTT,11.5,1.1825872,FALSE,Primitive erythroid lineage
sci3-me-707.TTCCATCTTTCCGTTCCTTA,11.5,1.3359402,FALSE,Primitive erythroid lineage
sci3-me-180.TTAATGAGCTCCGTTCGGAT,11.5,0.6768174,FALSE,Definitive erythroid lineage
sci3-me-570.TTCTGGCCTTTGCGAGGCA,11.5,2.0717395,FALSE,Definitive erythroid lineage
sci3-me-580.CTAACGACTTATCATGATC,11.5,1.8343373,FALSE,Definitive erythroid lineage


In [41]:
meta = meta[order(match(meta$sample, colnames(counts)))]

In [42]:
summary(meta$sample == colnames(counts))

   Mode    TRUE 
logical   25000 

In [28]:
head(colnames(counts))

[1] "sci3-me-409.GTATCGCATCCGCTCCGGC"  "sci3-me-459.GTCGGAGTTTAGACTTCTT" 
[3] "sci3-me-707.TTCCATCTTTCCGTTCCTTA" "sci3-me-180.TTAATGAGCTCCGTTCGGAT"
[5] "sci3-me-570.TTCTGGCCTTTGCGAGGCA"  "sci3-me-580.CTAACGACTTATCATGATC"

In [29]:
head(meta)

sample,development_stage,Size_Factor,detected_doublet,Main_cell_type
<chr>,<dbl>,<dbl>,<lgl>,<chr>
sci3-me-001.TAACGACTTTGGTACTGCCT,9.5,0.6458519,FALSE,Endothelial cells
sci3-me-001.ACGAGGTTTAACTGATCTT,13.5,0.6620719,FALSE,Definitive erythroid lineage
sci3-me-001.CTTAGCGGTCCTAGCGCCT,13.5,0.6443773,FALSE,Endothelial cells
sci3-me-001.CTACGGCATGCTAACTTGC,10.5,0.6959865,FALSE,Endothelial cells
sci3-me-002.CAAGGCGTTCGCCGCCTCC,13.5,1.0203870,FALSE,Definitive erythroid lineage
sci3-me-002.TCTATACCTTCAGGAGGAGA,11.5,2.2191943,FALSE,Endothelial cells


In [30]:
cao_meta = read.table(paste0(cao_in, 'meta.csv'), header = TRUE, sep = ",", stringsAsFactors = FALSE, comment.char = "$")
head(cao_meta)

,sample,development_stage,Size_Factor,detected_doublet,Main_cell_type,cell_id
,<chr>,<dbl>,<dbl>,<lgl>,<chr>,<int>
1,sci3-me-037.ACGCCATATGAGCATATGG,11.5,0.3185023,FALSE,Primitive erythroid lineage,113922
2,sci3-me-290.AAGGCTACTCATCAGAATG,11.5,0.3199768,FALSE,Definitive erythroid lineage,858173
3,sci3-me-442.TTGAGTCCTTATAGACGCA,11.5,1.5276314,FALSE,Endothelial cells,1296568
4,sci3-me-222.AGTCGCATTTTCTTCCGGT,11.5,4.6507234,FALSE,Endothelial cells,669478
5,sci3-me-187.GACTGACGTGAGCATCATT,11.5,3.0125009,FALSE,Definitive erythroid lineage,571245
6,sci3-me-572.TTGCGGTCTTAACTCAATT,11.5,0.8375431,FALSE,Primitive erythroid lineage,1648632


In [31]:
cao_meta = read.table(paste0(cao_in, 'meta.csv'), header = TRUE, sep = ",", stringsAsFactors = FALSE, comment.char = "$")

cao_sce = SingleCellExperiment(assays = list("counts" = counts))

In [32]:
cao_sce

class: SingleCellExperiment 
dim: 17573 25000 
metadata(0):
assays(1): counts
rownames(17573): Xkr4 Rp1 ... mt-Nd6 mt-Cytb
rowData names(0):
colnames(25000): sci3-me-409.GTATCGCATCCGCTCCGGC
  sci3-me-459.GTCGGAGTTTAGACTTCTT ... sci3-me-454.ATAAGCGAATTCGGCCTTAC
  sci3-me-111.TTCAACTGATGGTATCCGCC
colData names(0):
reducedDimNames(0):
mainExpName: NULL
altExpNames(0):

In [33]:
head(cao_meta)


,sample,development_stage,Size_Factor,detected_doublet,Main_cell_type,cell_id
,<chr>,<dbl>,<dbl>,<lgl>,<chr>,<int>
1,sci3-me-037.ACGCCATATGAGCATATGG,11.5,0.3185023,FALSE,Primitive erythroid lineage,113922
2,sci3-me-290.AAGGCTACTCATCAGAATG,11.5,0.3199768,FALSE,Definitive erythroid lineage,858173
3,sci3-me-442.TTGAGTCCTTATAGACGCA,11.5,1.5276314,FALSE,Endothelial cells,1296568
4,sci3-me-222.AGTCGCATTTTCTTCCGGT,11.5,4.6507234,FALSE,Endothelial cells,669478
5,sci3-me-187.GACTGACGTGAGCATCATT,11.5,3.0125009,FALSE,Definitive erythroid lineage,571245
6,sci3-me-572.TTGCGGTCTTAACTCAATT,11.5,0.8375431,FALSE,Primitive erythroid lineage,1648632


In [34]:
counts(cao_sce)[1:10, 1:10]

   [[ suppressing 10 column names ‘sci3-me-409.GTATCGCATCCGCTCCGGC’, ‘sci3-me-459.GTCGGAGTTTAGACTTCTT’, ‘sci3-me-707.TTCCATCTTTCCGTTCCTTA’ ... ]]



10 x 10 sparse Matrix of class "dgTMatrix"
                           
Xkr4    . . . . . . . . . .
Rp1     . . . . . . 1 . . .
Sox17   . . . . . . . . . .
Mrpl15  . . . . . . . . . .
Lypla1  . . . . . . . . . .
Gm37988 . . . . 1 . . . . .
Tcea1   . . . . . . . . . .
Rgs20   . . . . . . . . . .
Atp6v1h . . . . . . . . . .
Oprk1   . . . . . . . . . .

In [35]:
colData(cao_sce) = cao_meta %>% tibble::column_to_rownames("sample") %>% DataFrame

In [36]:
counts(cao_sce)[1:10, 1:10]

   [[ suppressing 10 column names ‘sci3-me-037.ACGCCATATGAGCATATGG’, ‘sci3-me-290.AAGGCTACTCATCAGAATG’, ‘sci3-me-442.TTGAGTCCTTATAGACGCA’ ... ]]



10 x 10 sparse Matrix of class "dgTMatrix"
                           
Xkr4    . . . . . . . . . .
Rp1     . . . . . . 1 . . .
Sox17   . . . . . . . . . .
Mrpl15  . . . . . . . . . .
Lypla1  . . . . . . . . . .
Gm37988 . . . . 1 . . . . .
Tcea1   . . . . . . . . . .
Rgs20   . . . . . . . . . .
Atp6v1h . . . . . . . . . .
Oprk1   . . . . . . . . . .

In [37]:
cao_sce
head(colData(cao_sce))

class: SingleCellExperiment 
dim: 17573 25000 
metadata(0):
assays(1): counts
rownames(17573): Xkr4 Rp1 ... mt-Nd6 mt-Cytb
rowData names(0):
colnames(25000): sci3-me-037.ACGCCATATGAGCATATGG
  sci3-me-290.AAGGCTACTCATCAGAATG ... sci3-me-324.TAACTAAGGTATCGAGTCGC
  sci3-me-743.ATGGTAACTTAAGCCGGCTG
colData names(5): development_stage Size_Factor detected_doublet
  Main_cell_type cell_id
reducedDimNames(0):
mainExpName: NULL
altExpNames(0):

DataFrame with 6 rows and 5 columns
                                development_stage Size_Factor detected_doublet
                                        <numeric>   <numeric>        <logical>
sci3-me-037.ACGCCATATGAGCATATGG              11.5    0.318502            FALSE
sci3-me-290.AAGGCTACTCATCAGAATG              11.5    0.319977            FALSE
sci3-me-442.TTGAGTCCTTATAGACGCA              11.5    1.527631            FALSE
sci3-me-222.AGTCGCATTTTCTTCCGGT              11.5    4.650723            FALSE
sci3-me-187.GACTGACGTGAGCATCATT              11.5    3.012501            FALSE
sci3-me-572.TTGCGGTCTTAACTCAATT              11.5    0.837543            FALSE
                                        Main_cell_type   cell_id
                                           <character> <integer>
sci3-me-037.ACGCCATATGAGCATATGG Primitive erythroid ..    113922
sci3-me-290.AAGGCTACTCATCAGAATG Definitive erythroid..    858173
sci3-me-442.TTGAGTCCTTATAGACGCA      Endothelial cells   1296568
sci3-me

In [6]:
head(cells)

[1] "sci3-me-409.GTATCGCATCCGCTCCGGC"  "sci3-me-459.GTCGGAGTTTAGACTTCTT" 
[3] "sci3-me-707.TTCCATCTTTCCGTTCCTTA" "sci3-me-180.TTAATGAGCTCCGTTCGGAT"
[5] "sci3-me-570.TTCTGGCCTTTGCGAGGCA"  "sci3-me-580.CTAACGACTTATCATGATC"

In [ ]:
atlas_sce <- computeSumFactors(atlas_sce)
atlas_sce <- logNormCounts(atlas_sce)